# 가까운 무릎·발목 첫 파인튜닝

기존 사람별 분할에서 학습 5장과 기본 검증 5장을 사용한다. 오른쪽 무릎 보호대와 발목 테이핑이 있는 **a05 41번**은 별도 평가한다. 같은 프레임을 원본 모델과 학습 후 모델에 넣어 수행자를 놓친 횟수와 관절 위치 오차를 비교한다.

처음에는 **모든 셀 실행**으로 원본 라벨과 학습 전 예측을 본다. 그림을 확인한 뒤 **4단계의 학습 스위치**를 켜고 4·5단계 셀을 차례로 실행한다.

### 1. 수동 판독과 분할 불러오기

학습·검증에 쓰는 사람과 예비 평가 대상을 표로 확인한다. 데이터셋에는 COCO 순서의 17개 관절 칸을 유지한다. 가까운 무릎·발목 직접점만 가시성 2로 기록한다. 다른 15점은 위치 손실에서 빠지며 가시성 학습에는 0 표적으로 쓰인다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display
from ultralytics import YOLO

from wlcv.config import DEVICE, ROOT
from wlcv.data.pose_dataset import load_finetune_frames, export_pose_dataset, check_finetune_files
from wlcv.evaluation.pose_label_review import review_pose_labels
from wlcv.evaluation.pose_comparison import evaluate_pose, detection_summary, paired_errors, comparison_figure
from wlcv.training.pose import ORIGINAL_POSE, train_pose

# 커널을 다시 열어 결과만 볼 때는 4단계가 출력한 실행 폴더를 지정한다.
EXISTING_RUN = None  # 예: ROOT / "outputs/runs/pose_first_finetune"
groups = load_finetune_frames()
display(pd.DataFrame([{"용도": role, "영상": s.video.stem, "프레임": s.frame, "가까운 쪽": s.side}
                      for role, samples in groups.items() for s in samples]))
print("시작 가중치", ORIGINAL_POSE.resolve(strict=True))
print("Mac 학습 장치", DEVICE)

### 2. 실제 학습 입력을 원본에 겹쳐 보기

학습 5장과 기본 검증 5장을 PNG와 YOLO TXT로 내보낸다. 예비 프레임은 학습 YAML·검증 로더·최고 가중치 선택에 사용하지 않는다. 저장된 TXT를 원본 픽셀로 복원해 박스·무릎·발목이 CSV와 0.01픽셀 이내에서 일치하는지 확인한다. 그림에서 초록 선은 박스, 청록 십자는 무릎, 노란 마름모는 발목이다.

In [ ]:
verified_data = None
baseline_data = None
run_dir = Path(EXISTING_RUN).expanduser().resolve() if EXISTING_RUN else None
if run_dir is None:
    data_yaml = export_pose_dataset(groups["train"] + groups["val"], ROOT / "outputs/pose_first_finetune")
else:
    args = yaml.safe_load((run_dir / "args.yaml").read_text())
    if args["task"] != "pose" or Path(args["model"]).resolve() != ORIGINAL_POSE.resolve():
        raise ValueError("원본 YOLO26n-pose에서 시작한 실행 폴더를 지정하세요.")
    data_yaml = Path(args["data"]).resolve(strict=True)

check_finetune_files(data_yaml, groups)
for role in ["train", "val"]:
    print(role, "라벨")
    figure, errors = review_pose_labels(data_yaml, groups[role])
    plt.show()
    display(errors.round(5))
verified_data = data_yaml
print("확인한 데이터셋", verified_data)

### 3. 원본 모델의 수행자와 가까운 관절 확인

두 모델 모두 같은 원본, CPU, 입력 크기 640을 사용한다. **수행자 미탐**은 사람 신뢰도 0.25를 통과한 후보 중 수동 수행자 박스와 IoU가 0.50 이상인 사람이 없는 경우다. 여러 명이면 IoU가 가장 높은 사람을 대응시킨다.

관절은 해당 선수의 가까운 쪽에서 신뢰도 0.50 이상인 유효 좌표만 측정한다. 수동점과 예측점 사이 직선거리를 픽셀로 구하고, 이를 **수동 수행자 박스 높이**로 나눠 백분율로도 표시한다. 표의 빈 오차는 미탐·저신뢰·좌표 없음 중 어떤 경우인지 상태 열에서 확인한다.

그림의 초록 실선은 수동 박스, 주황 점선은 가장 많이 겹친 후보 박스다. 청록은 무릎, 노랑은 발목이며 동그라미는 수동점, ×는 측정 가능한 예측점이다. MISS는 수행자 미탐을 표시한다.

In [ ]:
eval_groups = {"기본 검증": groups["val"], "예비 a05": groups["reserve"]}
filenames = {"기본 검증": "validation", "예비 a05": "reserve_a05"}
baseline_data = None
before = {}
original = YOLO(str(ORIGINAL_POSE))
before_folder = data_yaml.parent / "baseline_evaluation"
before_folder.mkdir(exist_ok=True)
for title, samples in eval_groups.items():
    print(title)
    table = evaluate_pose(original, samples)
    before[title] = table
    table.to_csv(before_folder / f"{filenames[title]}.csv", index=False)
    display(detection_summary({"원본": table}))
    display(table[["영상", "프레임", "관절", "상태", "후보 IoU", "관절 신뢰도", "오차(px)", "박스 높이 대비(%)"]].round(2))
    figure = comparison_figure(samples, {"Before": table})
    figure.savefig(before_folder / f"{filenames[title]}.png", dpi=140)
    plt.show()
baseline_data = data_yaml
print("학습 전 평가 폴더", before_folder)

### 4. Mac에서 첫 파인튜닝 실행

첫 실행은 **20 epochs**로 둔다. 학습 프레임이 다섯 장이므로 짧게 시작해 미탐과 위치 오차가 어떻게 달라지는지 보는 설정이다. 시험 학습 때와 같이 한 장씩 읽고(`batch=1`) 현재 프로세스를 사용한다(`workers=0`). `nbs=5`로 두어 워밍업이 끝난 뒤 약 다섯 장 분량의 기울기를 모아 한 번 갱신한다. 옵티마이저와 학습률은 설치된 Ultralytics의 자동 설정을 따른다.

검수 후 `RUN_TRAINING = True`로 바꾸고 이 셀을 실행한다. 시작점은 `models/yolo26n-pose.pt`다. 완료되면 실행 폴더와 최고·마지막 가중치가 표시된다.

In [ ]:
RUN_TRAINING = False
EPOCHS = 20

if RUN_TRAINING:
    if EXISTING_RUN or verified_data != data_yaml or baseline_data != data_yaml:
        raise ValueError("새 학습은 EXISTING_RUN을 None으로 두고 1~3단계를 실행한 뒤 시작하세요.")
    run_dir = train_pose(verified_data, epochs=EPOCHS, name="pose_first_finetune", device=DEVICE, nbs=5)
if run_dir is not None:
    print("이번 실행 폴더", run_dir)
    for name in ["best.pt", "last.pt"]:
        print((run_dir / "weights" / name).resolve(strict=True))
    display(pd.read_csv(run_dir / "results.csv").tail(1))
else:
    print("원본 평가 완료. 학습을 시작하려면 이 셀의 스위치를 켜세요.")

### 5. 같은 검증 프레임에서 원본과 학습 후 비교

기본 검증 5장으로 Ultralytics가 선택한 `best.pt`를 다시 연다. 먼저 수행자 미탐 수와 측정 가능한 관절 수를 읽는다. 다음 표에서 **두 모델 모두 측정한 같은 관절**끼리 ‘학습 후 − 학습 전’을 계산한다. 음수는 위치 오차 감소다. 한 모델이 놓친 행은 상태를 남기고 변화량을 비워 둔다. 예비 a05는 별도 표와 이미지로 저장한다.

기본 검증 5장은 가중치 선택에도 사용된다. 이 표는 현재 프레임에서 일어난 변화를 해석하는 데 사용한다.

In [ ]:
if run_dir is None:
    print("4단계 학습이 끝나면 이 셀을 실행하세요. 커널을 다시 열었다면 1단계의 EXISTING_RUN을 지정하세요.")
else:
    if baseline_data != data_yaml:
        raise ValueError("이번 데이터의 원본 평가를 3단계에서 먼저 실행하세요.")
    best = YOLO(str((run_dir / "weights/best.pt").resolve(strict=True)))
    report = run_dir / "manual_evaluation"
    report.mkdir(exist_ok=True)
    for title, samples in eval_groups.items():
        print(title)
        after = evaluate_pose(best, samples)
        models = {"Before": before[title], "After": after}
        counts = detection_summary(models)
        detail, summary = paired_errors(before[title], after)
        display(counts, detail.round(2), summary.round(2))
        prefix = filenames[title]
        for suffix, table in [("before", before[title]), ("after", after), ("detection", counts),
                              ("paired", detail), ("change", summary.reset_index())]:
            table.to_csv(report / f"{prefix}_{suffix}.csv", index=False)
        figure = comparison_figure(samples, models)
        figure.savefig(report / f"{prefix}_comparison.png", dpi=140)
        plt.show()
    print("전후 비교 표와 이미지", report)